In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [3]:
data

,CS textfield,BB textfield,Stack,Head angle,Head tube length textfield,Seat stay junction0,Seat tube length,Seat angle,DT Length,FORK0R,...,BELTorCHAIN OHCLASS: 1,RIM_STYLE front OHCLASS: DISC,RIM_STYLE front OHCLASS: SPOKED,RIM_STYLE front OHCLASS: TRISPOKE,RIM_STYLE rear OHCLASS: DISC,RIM_STYLE rear OHCLASS: SPOKED,RIM_STYLE rear OHCLASS: TRISPOKE,Seat tube type OHCLASS: 0,Seat tube type OHCLASS: 1,Seat tube type OHCLASS: 2
3553,381.00,55.002477,520.518278,72.000376,100.0,100.0,500.0,75.000376,619.064231,43.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1780,480.00,0.000000,530.900122,73.000000,152.7,45.0,489.8,90.000000,608.123819,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2322,425.00,62.000000,598.475325,72.200000,160.3,45.0,560.0,73.700000,657.859061,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1358,400.00,-37.500000,390.902939,67.500000,150.0,125.0,340.0,67.500000,469.378527,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
840,425.29,70.000000,571.512441,72.000000,124.0,45.0,565.8,74.300000,634.843371,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,430.00,65.000000,584.677620,72.000000,124.8,60.0,541.5,71.800000,632.625640,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4477,410.00,70.000000,658.278407,73.000000,226.4,55.0,610.0,73.000000,679.431932,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4659,420.00,72.000000,617.038218,72.500000,170.3,45.0,577.9,72.310741,659.860913,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1756,374.00,0.000000,482.702362,73.000000,115.0,55.0,301.0,74.000000,607.721727,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [4]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [5]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")

condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [6]:
eval_scores = evaluator(data_tens, condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [7]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [8]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Calculating reference point for scoring functions...
Calculating reference point for scoring functions...


c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [9]:
main_scorer(data_tens, condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Hypervolume                     0.000000
Constraint Satisfaction Rate    0.839286
Maximum Mean Discrepancy        0.003201
dtype: float64

In [10]:
detailed_scorer(data_tens, condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Min Objective Score: Drag Force                                                                               7.652876
Min Objective Score: Knee Angle Error                                                                       341.780884
Min Objective Score: Hip Angle Error                                                                         72.571136
Min Objective Score: Arm Angle Error                                                                         97.584274
Min Objective Score: Mass                                                                                    20.529053
Min Objective Score: Planar Compliance                                                                      209.664429
Min Objective Score: Transverse Compliance                                                                  295.839478
Min Objective Score: Eccentric Compliance                                                                   135.624603
Mean Objective Score: Drag Force                